# National Real Estate Prices (Python Notebook)

This notebook mirrors the original `Final Project.Rmd` flow chunk-by-chunk using Python.

## 1) Read data
Equivalent to the first R chunk (`read.csv` calls). This notebook assumes files are in `data/`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from scipy.stats import linregress
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists() and (BASE_DIR / "national-real-estate-prices-python").exists():
    BASE_DIR = BASE_DIR / "national-real-estate-prices-python"

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cpi = pd.read_csv(DATA_DIR / "CPI-U.csv")
mortgage_rates = pd.read_csv(DATA_DIR / "30_year_mortgate_rates.csv")
housing_usa = pd.read_csv(DATA_DIR / "Housing_Index_USA.csv")
housing_mountain = pd.read_csv(DATA_DIR / "Housing_Index_Mountain.csv")
housing_colorado = pd.read_csv(DATA_DIR / "Housing_Index_Colorado.csv")
housing_denver = pd.read_csv(DATA_DIR / "Housing_Index_Denver.csv")
housing_boulder = pd.read_csv(DATA_DIR / "Housing_Index_Boulder.csv")

for name, frame in {
    "CPI": cpi,
    "Mortgage_Rates": mortgage_rates,
    "Housing_USA": housing_usa,
    "Housing_Mountain": housing_mountain,
    "Housing_Colorado": housing_colorado,
    "Housing_Denver": housing_denver,
    "Housing_Boulder": housing_boulder,
}.items():
    print(f"{name}: shape={frame.shape}")
    display(frame.head(3))

## 2) Convert to fractional year, visualize, merge monthly USA/Mountain + CPI + Mortgage
Equivalent to the large ggplot/merge/scatter R chunk.

In [ ]:
# Convert Year/Month to FractionalYear
cpi["FractionalYear"] = cpi["Year"] + cpi["Month"] / 12
mortgage_rates["FractionalYear"] = mortgage_rates["Year"] + mortgage_rates["Month"] / 12
housing_usa["FractionalYear"] = housing_usa["Year"] + housing_usa["Month"] / 12
housing_mountain["FractionalYear"] = housing_mountain["Year"] + housing_mountain["Month"] / 12

# Convert Year/Quarter to FractionalYear
housing_colorado["FractionalYear"] = housing_colorado["Year"] + housing_colorado["Quarter"] / 4
housing_denver["FractionalYear"] = housing_denver["Year"] + housing_denver["Quarter"] / 4
housing_boulder["FractionalYear"] = housing_boulder["Year"] + housing_boulder["Quarter"] / 4

# Time-series plots
plot_specs = [
    (cpi, "Index", "CPI over Time", "line_cpi_notebook.png"),
    (mortgage_rates, "Rate", "30-Year Fixed Mortgage Rates over Time", "line_mortgage_notebook.png"),
    (housing_usa, "Index", "Housing Price Index (USA) over Time", "line_hpi_usa_notebook.png"),
    (housing_mountain, "Index", "Housing Price Index (Mountain) over Time", "line_hpi_mountain_notebook.png"),
    (housing_colorado, "Index", "Housing Price Index (Colorado) over Time", "line_hpi_colorado_notebook.png"),
    (housing_denver, "Index", "Housing Price Index (Denver) over Time", "line_hpi_denver_notebook.png"),
    (housing_boulder, "Index", "Housing Price Index (Boulder) over Time", "line_hpi_boulder_notebook.png"),
]

for frame, y_col, title, fname in plot_specs:
    plt.figure(figsize=(9, 4.5))
    sns.lineplot(data=frame, x="FractionalYear", y=y_col)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / fname, dpi=140)
    plt.show()

# Merge CPI + USA + Mountain + Mortgage
df = cpi.merge(housing_usa, on="FractionalYear", suffixes=("_CPI", "_USA"))
df = df.merge(housing_mountain, on="FractionalYear", suffixes=("", "_MOUNTAIN"))
df = df.merge(mortgage_rates, on="FractionalYear", suffixes=("", "_MORT"))
df = df[["FractionalYear", "Index_CPI", "Index_USA", "Index", "Rate"]].copy()
df.columns = ["Year", "CPI", "HousingIndex_USA", "HousingIndex_Mountain", "Mortgage_Rate"]

# Scatter (USA + Mountain vs CPI)
plt.figure(figsize=(8, 6))
plt.scatter(df["CPI"], df["HousingIndex_Mountain"], s=16, alpha=0.75, label="Housing Index (Mountain)", color="red")
plt.scatter(df["CPI"], df["HousingIndex_USA"], s=16, alpha=0.75, label="Housing Index (USA)", color="blue")
plt.xlabel("Consumer Price Index")
plt.ylabel("Housing Index")
plt.legend()
plt.title("Housing Indexes vs CPI")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "scatter_housing_usa_mountain_vs_cpi_notebook.png", dpi=140)
plt.show()

## 3) Merge CPI + Mortgage into `df1` (monthly)
Equivalent to the R chunk creating `df1`.

In [ ]:
df1 = cpi.merge(mortgage_rates, on="FractionalYear", suffixes=("_CPI", "_MORT"))
df1 = df1[["FractionalYear", "Index", "Rate"]].copy()
df1.columns = ["Year", "CPI", "Mortgage_Rate"]
display(df1.head())

## 4) Build `df2` (quarterly Colorado, Denver, Boulder + CPI + Mortgage)
Equivalent to the R chunk creating/scattering `df2`.

In [ ]:
df2 = cpi.merge(housing_colorado, on="FractionalYear", suffixes=("_CPI", "_CO"))
df2 = df2.merge(housing_denver, on="FractionalYear", suffixes=("", "_DEN"))
df2 = df2.merge(housing_boulder, on="FractionalYear", suffixes=("", "_BOU"))
df2 = df2.merge(mortgage_rates, on="FractionalYear", suffixes=("", "_MORT"))
df2 = df2[["FractionalYear", "Index_CPI", "Index_CO", "Index", "Index_BOU", "Rate"]].copy()
df2.columns = ["Year", "CPI", "HousingIndex_Colorado", "HousingIndex_Denver", "HousingIndex_Boulder", "Mortgage_Rate"]

plt.figure(figsize=(8, 6))
plt.scatter(df2["CPI"], df2["HousingIndex_Colorado"], s=18, alpha=0.75, label="Housing Index Colorado", color="red")
plt.scatter(df2["CPI"], df2["HousingIndex_Denver"], s=18, alpha=0.75, label="Housing Index Denver", color="blue")
plt.scatter(df2["CPI"], df2["HousingIndex_Boulder"], s=18, alpha=0.75, label="Housing Index Boulder", color="green")
plt.xlabel("Consumer Price Index")
plt.ylabel("Housing Index")
plt.legend()
plt.title("Housing Indexes (CO/Denver/Boulder) vs CPI")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "scatter_housing_co_denver_boulder_vs_cpi_notebook.png", dpi=140)
plt.show()

## 5) Derive inflation rates and build `df3`
Equivalent to the long inflation loop chunk in R.

In [ ]:
def annualized_inflation(series: pd.Series, periods_per_year: int) -> pd.Series:
    values = series.to_numpy(dtype=float)
    out = np.full(len(values), np.nan)
    for i in range(len(values)):
        if i < periods_per_year:
            if i + 1 < len(values):
                out[i] = periods_per_year * (values[i + 1] / values[i] - 1.0) * 100.0
        else:
            out[i] = (values[i] / values[i - periods_per_year] - 1.0) * 100.0
    return pd.Series(out, index=series.index)

# df monthly inflation
df["Consumer_Inflation_Rate"] = annualized_inflation(df["CPI"], 12)
df["House_Inflation_Rate_USA"] = annualized_inflation(df["HousingIndex_USA"], 12)
df["House_Inflation_Rate_Mountain"] = annualized_inflation(df["HousingIndex_Mountain"], 12)

# df1 monthly inflation
df1["Consumer_Inflation_Rate"] = annualized_inflation(df1["CPI"], 12)
df1 = df1[["Year", "Mortgage_Rate", "Consumer_Inflation_Rate"]]

# df2 quarterly inflation
df2["Consumer_Inflation_Rate"] = annualized_inflation(df2["CPI"], 4)
df2["House_Inflation_Rate_Colorado"] = annualized_inflation(df2["HousingIndex_Colorado"], 4)
df2["House_Inflation_Rate_Denver"] = annualized_inflation(df2["HousingIndex_Denver"], 4)
df2["House_Inflation_Rate_Boulder"] = annualized_inflation(df2["HousingIndex_Boulder"], 4)

# merge df2 + df => df3
df3 = df2.merge(df, on="Year", suffixes=("", "_monthly"))

display(df.head(3))
display(df1.head(3))
display(df2.head(3))
display(df3.head(3))

## 6) Correlation views (replacement for `chart.Correlation`)
Equivalent to the 3 correlation-chart R chunks.

In [ ]:
def correlation_views(frame: pd.DataFrame, cols: list[str], label: str):
    corr = frame[cols].corr(numeric_only=True)
    display(corr)

    plt.figure(figsize=(7.5, 6))
    sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
    plt.title(f"Correlation heatmap: {label}")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{label}_corr_heatmap_notebook.png", dpi=140)
    plt.show()

    g = sns.pairplot(frame[cols].dropna(), corner=True, diag_kind="hist", plot_kws={"s": 15, "alpha": 0.65})
    g.fig.suptitle(f"Pair plot: {label}", y=1.02)
    g.savefig(OUTPUT_DIR / f"{label}_pairplot_notebook.png", dpi=140)
    plt.show()

correlation_views(df, ["Consumer_Inflation_Rate", "Mortgage_Rate", "House_Inflation_Rate_USA", "House_Inflation_Rate_Mountain"], "df_monthly")
correlation_views(df2, ["Consumer_Inflation_Rate", "Mortgage_Rate", "House_Inflation_Rate_Colorado", "House_Inflation_Rate_Denver", "House_Inflation_Rate_Boulder"], "df2_quarterly")
correlation_views(df3, ["Consumer_Inflation_Rate", "Mortgage_Rate", "House_Inflation_Rate_Colorado", "House_Inflation_Rate_Denver", "House_Inflation_Rate_Boulder", "House_Inflation_Rate_USA", "House_Inflation_Rate_Mountain"], "df3_merged")

## 7) Individual correlations/regressions for `df` (monthly)
Equivalent to the R chunk titled *Analyze individual correlations for 1991-2022 data (df, monthly)*.

In [ ]:
def reg_plot(frame: pd.DataFrame, x_col: str, y_col: str, title: str, file_name: str):
    clean = frame[[x_col, y_col]].dropna()
    fit = linregress(clean[x_col], clean[y_col])

    plt.figure(figsize=(7.5, 5.5))
    sns.scatterplot(data=clean, x=x_col, y=y_col, s=20)
    xs = np.linspace(clean[x_col].min(), clean[x_col].max(), 120)
    ys = fit.intercept + fit.slope * xs
    plt.plot(xs, ys, color="red", linewidth=2)
    plt.title(title)
    plt.text(0.03, 0.95, f"r={fit.rvalue:.2f}, p={fit.pvalue:.3g}", transform=plt.gca().transAxes, va="top")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / file_name, dpi=140)
    plt.show()

    print({"slope": fit.slope, "intercept": fit.intercept, "r": fit.rvalue, "p": fit.pvalue, "n": len(clean)})

reg_plot(df, "Consumer_Inflation_Rate", "House_Inflation_Rate_USA", "House Inflation Rate (USA) vs Consumer Inflation", "nb_reg_df_house_usa_vs_cpi.png")
reg_plot(df, "Consumer_Inflation_Rate", "House_Inflation_Rate_Mountain", "House Inflation Rate (Mountain) vs Consumer Inflation", "nb_reg_df_house_mountain_vs_cpi.png")
reg_plot(df, "House_Inflation_Rate_USA", "House_Inflation_Rate_Mountain", "Mountain vs USA House Inflation", "nb_reg_df_house_mountain_vs_usa.png")
reg_plot(df, "Consumer_Inflation_Rate", "Mortgage_Rate", "Mortgage Rate vs Consumer Inflation (monthly)", "nb_reg_df_mortgage_vs_cpi.png")
reg_plot(df, "Mortgage_Rate", "House_Inflation_Rate_USA", "USA House Inflation vs Mortgage Rate", "nb_reg_df_house_usa_vs_mortgage.png")

## 8) Individual correlations/regressions for `df2` (quarterly)
Equivalent to the R chunk for quarterly regional analysis.

In [ ]:
reg_plot(df2, "Consumer_Inflation_Rate", "House_Inflation_Rate_Boulder", "Boulder House Inflation vs Consumer Inflation", "nb_reg_df2_boulder_vs_cpi.png")
reg_plot(df2, "Consumer_Inflation_Rate", "House_Inflation_Rate_Colorado", "Colorado House Inflation vs Consumer Inflation", "nb_reg_df2_colorado_vs_cpi.png")
reg_plot(df2, "House_Inflation_Rate_Colorado", "House_Inflation_Rate_Boulder", "Boulder vs Colorado House Inflation", "nb_reg_df2_boulder_vs_colorado.png")
reg_plot(df2, "Consumer_Inflation_Rate", "Mortgage_Rate", "Mortgage Rate vs Consumer Inflation (quarterly)", "nb_reg_df2_mortgage_vs_cpi.png")
reg_plot(df2, "Mortgage_Rate", "House_Inflation_Rate_Boulder", "Boulder House Inflation vs Mortgage Rate", "nb_reg_df2_boulder_vs_mortgage.png")

## 9) Individual correlation/regression for `df1` (long monthly)
Equivalent to the R chunk for 1971+ mortgage vs inflation.

In [ ]:
reg_plot(df1, "Consumer_Inflation_Rate", "Mortgage_Rate", "Mortgage Rate vs Consumer Inflation (long monthly)", "nb_reg_df1_mortgage_vs_cpi.png")

## 10) K-means clustering on `df3` (all indexes)
Equivalent to first clustering chunk (`k=4,6,5`).

In [ ]:
def run_kmeans_plot(frame: pd.DataFrame, cols: list[str], k: int, label: str):
    clean = frame[cols].dropna().reset_index(drop=True)
    scaler = StandardScaler()
    x = scaler.fit_transform(clean)

    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    cluster = km.fit_predict(x)

    pca = PCA(n_components=2, random_state=42)
    x2 = pca.fit_transform(x)

    clustered = clean.copy()
    clustered["cluster"] = cluster
    display(clustered.head())

    plt.figure(figsize=(7.5, 5.5))
    sns.scatterplot(x=x2[:, 0], y=x2[:, 1], hue=cluster, palette="tab10", s=26)
    plt.title(f"{label}: k={k} (PCA view)")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"nb_{label}_k{k}.png", dpi=140)
    plt.show()
    return clustered

all_cols_df3 = [
    "CPI", "HousingIndex_Colorado", "HousingIndex_Denver", "HousingIndex_Boulder", "Mortgage_Rate",
    "Consumer_Inflation_Rate", "House_Inflation_Rate_Colorado", "House_Inflation_Rate_Denver",
    "House_Inflation_Rate_Boulder", "House_Inflation_Rate_USA", "House_Inflation_Rate_Mountain"
]

clusterGroups4 = run_kmeans_plot(df3, all_cols_df3, 4, "kmeans_df3_all_indexes")
clusterGroups6 = run_kmeans_plot(df3, all_cols_df3, 6, "kmeans_df3_all_indexes")
clusterGroups5 = run_kmeans_plot(df3, all_cols_df3, 5, "kmeans_df3_all_indexes")

## 11) K-means clustering on `df4` (inflation-only subset)
Equivalent to second clustering chunk (`k=4,5,6`).

In [ ]:
df4 = df3[[
    "Year", "Mortgage_Rate", "Consumer_Inflation_Rate", "House_Inflation_Rate_Colorado",
    "House_Inflation_Rate_Denver", "House_Inflation_Rate_Boulder", "House_Inflation_Rate_USA",
    "House_Inflation_Rate_Mountain"
]].copy()

clusterInflationGroups4 = run_kmeans_plot(df4, df4.columns.tolist(), 4, "kmeans_df4_inflation")
clusterInflationGroups5 = run_kmeans_plot(df4, df4.columns.tolist(), 5, "kmeans_df4_inflation")
clusterInflationGroups6 = run_kmeans_plot(df4, df4.columns.tolist(), 6, "kmeans_df4_inflation")

## 12) Analyze cluster-specific regressions for `df4` k=4 solution
Equivalent to the R chunk analyzing each cluster separately.

In [ ]:
for cluster_id in sorted(clusterInflationGroups4["cluster"].unique()):
    cluster_df = clusterInflationGroups4[clusterInflationGroups4["cluster"] == cluster_id]
    print(f"Cluster {cluster_id}: n={len(cluster_df)}")
    reg_plot(
        cluster_df,
        "Consumer_Inflation_Rate",
        "Mortgage_Rate",
        f"Mortgage vs Consumer Inflation (df4 cluster {cluster_id})",
        f"nb_cluster_df4_{cluster_id}_mortgage_vs_cpi.png",
    )
    print("Years:")
    print(cluster_df["Year"].to_list())

## 13) K-means clustering for `df1` (mortgage + consumer inflation)
Equivalent to final clustering chunk (`k=4,5,6,3`).

In [ ]:
clusterCPIvsMortgageInflationGroups4 = run_kmeans_plot(df1, ["Year", "Mortgage_Rate", "Consumer_Inflation_Rate"], 4, "kmeans_df1_mortgage_cpi")
clusterCPIvsMortgageInflationGroups5 = run_kmeans_plot(df1, ["Year", "Mortgage_Rate", "Consumer_Inflation_Rate"], 5, "kmeans_df1_mortgage_cpi")
clusterCPIvsMortgageInflationGroups6 = run_kmeans_plot(df1, ["Year", "Mortgage_Rate", "Consumer_Inflation_Rate"], 6, "kmeans_df1_mortgage_cpi")
clusterCPIvsMortgageInflationGroups3 = run_kmeans_plot(df1, ["Year", "Mortgage_Rate", "Consumer_Inflation_Rate"], 3, "kmeans_df1_mortgage_cpi")

## 14) Analyze cluster-specific regressions for `df1` k=3 solution
Equivalent to the final R chunk.

In [ ]:
for cluster_id in sorted(clusterCPIvsMortgageInflationGroups3["cluster"].unique()):
    icluster = clusterCPIvsMortgageInflationGroups3[clusterCPIvsMortgageInflationGroups3["cluster"] == cluster_id]
    print(f"Icluster {cluster_id}: n={len(icluster)}")
    reg_plot(
        icluster,
        "Consumer_Inflation_Rate",
        "Mortgage_Rate",
        f"Mortgage vs Consumer Inflation (df1 cluster {cluster_id})",
        f"nb_cluster_df1_{cluster_id}_mortgage_vs_cpi.png",
    )
    print("Years:")
    print(icluster["Year"].to_list())

## 15) Optional: Save key tables

In [ ]:
df.to_csv(OUTPUT_DIR / "nb_df_monthly.csv", index=False)
df1.to_csv(OUTPUT_DIR / "nb_df1_monthly.csv", index=False)
df2.to_csv(OUTPUT_DIR / "nb_df2_quarterly.csv", index=False)
df3.to_csv(OUTPUT_DIR / "nb_df3_merged.csv", index=False)
print("Saved notebook output tables to outputs/")